In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from graphviz import Digraph
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import pandas as pd
import numpy as np
from IPython.display import Image as IPImage, display
from PIL import Image   # para abrir e salvar imagens

## Fluxograma que tem bacia (classificada), encaminhamento, tipo de escoamento (grav/recalque). Legenda externa.

In [48]:
import os
import unicodedata
import pandas as pd
from graphviz import Digraph
from PIL import Image

def gerar_fluxograma_com_legenda(
    caminho_entrada,
    caminho_saida,
    graphviz_bin_path,
    nome_arquivo,
    caminho_legenda=None,
    posicao_norm=(0.75, 0.05),  # posição normalizada (0..1) da legenda (topo-esquerda)
    escala=0.20,                # fator proporcional da legenda
    ref_dim="largura",          # "largura" | "altura" | "menor"
    criar_semLegenda = True
):
    """
    Gera fluxograma com cores pastel claras baseadas na coluna 'classificacao'
    e, se fornecida, sobrepõe uma imagem de legenda mantendo proporção.
    Resolve o problema de nós com cor "perdida" definindo todos os nós antes das arestas
    e normalizando textos (acentos/maiúsculas/espaços).
    """

    # ---------- Utilidades ----------
    def _norm(s):
        """Normaliza string: remove acentos, aparas espaços e colapsa múltiplos espaços."""
        if s is None or (isinstance(s, float) and pd.isna(s)):
            return ""
        s = str(s).strip()
        s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
        s = " ".join(s.split())
        return s

    # Paleta de cores pastel clarinhas (expanda se quiser)
    paleta_pastel = [
        "#FFDAB9", "#C5E1A5", "#fdff73","#60a6a8", "#F7CAC9", 
        "#F9E79F", "#D7BDE2", "#cf4c4c", "#F6C1C1", "#FFE5B4",
        "#E2F0CB", "#FADADD", "#D0ECE7", "#F9CB9C", "#c9b685"
    ]

    # ---------- Entrada ----------
    os.environ["PATH"] += os.pathsep + graphviz_bin_path
    df = pd.read_excel(caminho_entrada)

    # Checagem de colunas obrigatórias
    for col in ["Bacias", "bacia_destino", "Tipo"]:
        if col not in df.columns:
            raise ValueError(f"Coluna obrigatória ausente: {col}")

    # Normalização
    df["Bacias"] = df["Bacias"].map(_norm)
    df["bacia_destino"] = df["bacia_destino"].map(_norm)
    if "classificacao" in df.columns:
        df["classificacao"] = df["classificacao"].map(lambda x: _norm(x) or None)
    if "classificacao_destino" in df.columns:
        df["classificacao_destino"] = df["classificacao_destino"].map(lambda x: _norm(x) or None)

    # ---------- Mapa de cores por classe ----------
        # ---------- Mapa de cores por classe ----------
    cor_leste = "#add8e6"   # ajuste aqui o azul
    cor_oeste = "#eb9d68"   # ajuste aqui o vermelho

    classes_origem = [c for c in df.get("classificacao", pd.Series(dtype=object)).dropna().unique()]
    classes_destino = [c for c in df.get("classificacao_destino", pd.Series(dtype=object)).dropna().unique()]
    classes = sorted(set(classes_origem) | set(classes_destino))

    cor_por_classe = {}
    for i, cls in enumerate(classes):
        cls_norm = _norm(cls).upper()
        if cls_norm == "LESTE":
            cor_por_classe[cls] = cor_leste
        elif cls_norm == "OESTE":
            cor_por_classe[cls] = cor_oeste
        else:
            cor_por_classe[cls] = paleta_pastel[i % len(paleta_pastel)]

    # Classe por nó (prioriza quando aparece como origem)
    classe_por_no = {}
    for _, r in df.iterrows():
        origem = r["Bacias"]
        cls = r.get("classificacao")
        if origem and cls and origem not in classe_por_no:
            classe_por_no[origem] = cls

    # Se houver classificação explícita para destinos, considera também
    if "classificacao_destino" in df.columns:
        for _, r in df.iterrows():
            dest = r["bacia_destino"]
            cls = r.get("classificacao_destino")
            if dest and cls and dest not in classe_por_no:
                classe_por_no[dest] = cls

    # ---------- Graphviz ----------
    dot = Digraph(
        comment="Fluxograma Alternativa",
        format="png",
        engine="dot",
        graph_attr={
            "rankdir": "LR",
            "splines": "polyline",
            "size": "13.5,8.5",
            "dpi": "300",
            "fontname": "Arial Bold",
            "fontsize": "14",
            "labelloc": "t",
            "nodesep": "0.2",
            "ranksep": "0.3",
        },
        node_attr={
            "fontname": "Arial Bold",
            "fontsize": "11",
            "shape": "box",
            "style": "filled",
            "fillcolor": "lightblue",
            "width": "0.6",
            "height": "0.3",
        },
        edge_attr={
            "fontname": "Arial",
            "fontsize": "10",
            "arrowsize": "0.8",
            "penwidth": "1.5",
        },
    )
    
    # 1) Cria TODOS os nós uma única vez, já com cor
    todos_nos = set(df["Bacias"].dropna()) | set(df["bacia_destino"].dropna())
    for no in sorted(n for n in todos_nos if n):
        up = no.upper()
        if up.startswith("ETE"):
            dot.node(no, shape="ellipse", fillcolor="#D9D9D9")  # cinza claro
            continue
        if up.startswith("INT"):
            dot.node(no, fillcolor="#ffffff")  # verde bem claro
            continue
        cls = classe_por_no.get(no)
        cor_no = cor_por_classe.get(cls, "#CFE8ED")  # azul pastel padrão
        dot.node(no, fillcolor=cor_no)

    # 2) Desenha as arestas (cores por 'Tipo')
    for _, row in df.iterrows():
        origem = row["Bacias"]
        destino = row["bacia_destino"] if row["bacia_destino"] else "ETE"
        tipo = _norm(row.get("Tipo", "")).upper()
        cor_aresta = {"GRAVIDADE": "blue", "RECALQUE": "red"}.get(tipo, "black")
        dot.edge(origem, destino, color=cor_aresta)

    # ---------- Render ----------
    os.makedirs(caminho_saida, exist_ok=True)
    base_prefix = os.path.join(caminho_saida, nome_arquivo)
    base_png = f"{base_prefix}.png"
    dot.render(base_prefix, view=False, cleanup=True)

    if criar_semLegenda == True:
        print(f"[Fluxograma sem legenda] {base_png}")

    # ---------- Sobreposição da legenda (opcional) ----------
    if not (caminho_legenda and os.path.exists(caminho_legenda)):
        print("[Aviso] Sem legenda (arquivo não encontrado).")
        return

    base = Image.open(base_png).convert("RGBA")
    legenda = Image.open(caminho_legenda).convert("RGBA")

    W, H = base.size
    lw, lh = legenda.size
    aspect = lw / lh if lh else 1.0

    if ref_dim == "altura":
        ref = H
    elif ref_dim == "menor":
        ref = min(W, H)
    else:
        ref = W

    target_w = int(max(1, round(ref * escala)))
    target_h = int(max(1, round(target_w / aspect)))
    target_w, target_h = min(target_w, W), min(target_h, H)

    legenda_rs = legenda.resize((target_w, target_h), Image.LANCZOS)

    x = int(round(W * posicao_norm[0]))
    y = int(round(H * posicao_norm[1]))
    x = max(0, min(x, W - target_w))
    y = max(0, min(y, H - target_h))

    base.paste(legenda_rs, (x, y), legenda_rs)
    saida_final = os.path.join(caminho_saida, f"{nome_arquivo}_com_legenda.png")
    base.save(saida_final)
    print(f"[Legenda] OK → {saida_final}")

    if not criar_semLegenda and os.path.exists(base_png):
        os.remove(base_png)


## Alternativa 01

In [58]:
gerar_fluxograma_com_legenda(
    caminho_entrada=r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\inputFluxograma_Alternativa01CP_22.04.26.xlsx',
    caminho_saida=r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul',
    graphviz_bin_path=r'C:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\EC-ESGOTO_2024-25\fluxograma\drivers\Graphviz-12.2.1-win64\bin',
    nome_arquivo='fluxograma_Alt1',
    caminho_legenda=r"C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_legenda.png",
    posicao_norm=(1, 0),
    escala=0.25,
    ref_dim="menor",
    criar_semLegenda = True
)

[Fluxograma sem legenda] C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_Alt1.png
[Legenda] OK → C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_Alt1_com_legenda.png


## Alternativa 02

In [57]:
gerar_fluxograma_com_legenda(
    caminho_entrada=r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\inputFluxograma_Alternativa02CP_23.04.26.xlsx',
    caminho_saida=r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul',
    graphviz_bin_path=r'C:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\EC-ESGOTO_2024-25\fluxograma\drivers\Graphviz-12.2.1-win64\bin',
    nome_arquivo='fluxograma_Alt2',
    caminho_legenda=r"C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_legenda.png",
    posicao_norm=(0, 1),
    escala=0.25,
    ref_dim="menor",
    criar_semLegenda = True
)

[Fluxograma sem legenda] C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_Alt2.png
[Legenda] OK → C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_Alt2_com_legenda.png


## Alternativa 03

In [60]:
gerar_fluxograma_com_legenda(
    caminho_entrada=r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\inputFluxograma_Alternativa03CP_23.04.26.xlsx',
    caminho_saida=r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul',
    graphviz_bin_path=r'C:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\EC-ESGOTO_2024-25\fluxograma\drivers\Graphviz-12.2.1-win64\bin',
    nome_arquivo='fluxograma_Alt3',
    caminho_legenda=r"C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_legenda.png",
    posicao_norm=(1, 1),
    escala=0.25,
    ref_dim="menor",
    criar_semLegenda = True
)

[Fluxograma sem legenda] C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_Alt3.png
[Legenda] OK → C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\fluxograma_Alt3_com_legenda.png
